# Bayesian Data Analysis (BDA3) — Chapter 8, Section 8.3
## Stratified Sampling & Poststratification: The 1988 CBS Election Survey

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saurabh2086/saurabh2086.github.io/blob/main/bda3/notebooks/ch8_stratified_pymc.ipynb)

This notebook implements the complete Bayesian analysis of the 1988 CBS News pre-election survey ($n = 1{,}447$) divided across 16 strata ($4 \text{ regions} \times 4 \text{ urbanization tiers}$).

### Core Objectives:
1. **Model Formulation:** Implement the independent Dirichlet-Multinomial (no-pooling) model for 3 candidate preference categories ($k=1$: Bush, $k=2$: Dukakis, $k=3$: Other/No opinion).
2. **Prior Sensitivity Analysis:** Compare the naive flat prior ($a_k = 1$, adding $48$ phantom voters nationwide) against Gelman's exact unstratified equivalent prior ($a_k = 1/16$, adding $3$ pseudo-voters nationwide).
3. **Finite-Population Poststratification:** Impute stratum proportions and compute the poststratified national Bush lead:
   $$\Delta = \sum_{j=1}^{16} W_j (\theta_{1j} - \theta_{2j})$$
4. **Hierarchical Partial Pooling (Option C):** Map the 3-category simplex to Cartesian log-odds $(\beta_{1j}, \beta_{2j})$, fit a bivariate Gaussian hierarchical model with LKJ Cholesky covariance, and demonstrate empirical shrinkage (Table 8.3).
5. **Model Checking:** Run Gelman's posterior predictive test on minimum stratum support to verify absence of over-shrinkage.


In [ ]:
# Auto-setup dependencies if running inside Google Colab
import sys
if "google.colab" in sys.modules:
    !pip install -q "pymc>=5.0.0" "arviz>=0.14.0"

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

print(f"PyMC Version: {pm.__version__}")
print(f"ArviZ Version: {az.__version__}")


### 1. Survey Data & Stratum Counts
We load the 16 strata with their observed counts $y_{kj}$, total interviews $n_j$, and census population weights $W_j = N_j / N$.


In [ ]:
# 16 Strata Observed Counts (Bush, Dukakis, Other)
counts = np.array([
    [14, 29, 4],    # 1: Northeast I
    [23, 22, 1],    # 2: Northeast II
    [78, 69, 20],   # 3: Northeast III
    [32, 36, 1],    # 4: Northeast IV
    [19, 23, 5],    # 5: Midwest I
    [42, 42, 10],   # 6: Midwest II
    [59, 45, 12],   # 7: Midwest III
    [80, 49, 16],   # 8: Midwest IV
    [12, 6, 3],     # 9: South I
    [45, 39, 12],   # 10: South II
    [51, 40, 8],    # 11: South III
    [101, 64, 17],  # 12: South IV
    [17, 16, 1],    # 13: West I
    [41, 27, 9],    # 14: West II
    [67, 46, 11],   # 15: West III
    [46, 30, 7]     # 16: West IV
])

strata_names = [
    "Northeast I (Rural)", "Northeast II", "Northeast III", "Northeast IV (Urban)",
    "Midwest I (Rural)", "Midwest II", "Midwest III", "Midwest IV (Urban)",
    "South I (Rural)", "South II", "South III", "South IV (Urban)",
    "West I (Rural)", "West II", "West III", "West IV (Urban)"
]

n_vec = counts.sum(axis=1)

# Census Population Weights (W_j = N_j / N)
W = np.array([
    0.032, 0.032, 0.115, 0.048,
    0.032, 0.065, 0.080, 0.100,
    0.015, 0.066, 0.068, 0.126,
    0.023, 0.053, 0.086, 0.057
])
W_norm = W / W.sum()

df_survey = pd.DataFrame({
    "Stratum": range(1, 17),
    "Name": strata_names,
    "Sample_n": n_vec,
    "Bush_y1": counts[:, 0],
    "Dukakis_y2": counts[:, 1],
    "Other_y3": counts[:, 2],
    "Bush_pct": (counts[:, 0] / n_vec).round(3),
    "Dukakis_pct": (counts[:, 1] / n_vec).round(3),
    "Census_W": W
})
df_survey


### 2. PyMC Model: Independent Dirichlet-Multinomial
We specify the conjugate model in PyMC:
- **Prior:** $\theta_j \sim \text{Dirichlet}(a_k)$ where $a_k = 1/16$
- **Likelihood:** $y_j \sim \text{Multinomial}(n_j, \theta_j)$
- **Poststratification:** $\Delta = \sum_{j=1}^{16} W_j (\theta_{1j} - \theta_{2j})$


In [ ]:
with pm.Model() as model_nopooling:
    # Prior: a_k = 1/16 per stratum (3 pseudo-voters total nationwide)
    a = np.full((16, 3), 1.0 / 16.0)
    theta = pm.Dirichlet("theta", a=a, shape=(16, 3))
    
    # Likelihood
    obs = pm.Multinomial("obs", n=n_vec, p=theta, observed=counts)
    
    # Poststratified National Proportions
    national_bush = pm.Deterministic("national_bush", pm.math.dot(W_norm, theta[:, 0]))
    national_dukakis = pm.Deterministic("national_dukakis", pm.math.dot(W_norm, theta[:, 1]))
    national_lead = pm.Deterministic("national_lead", national_bush - national_dukakis)
    
    # Sample posterior
    idata = pm.sample(draws=3000, tune=1500, chains=4, random_seed=42, return_inferencedata=True)

az.summary(idata, var_names=["national_bush", "national_dukakis", "national_lead"])


In [ ]:
# Compare with Flat Prior a_k = 1.0 (48 phantom voters nationwide)
with pm.Model() as model_flat:
    a_flat = np.ones((16, 3))
    theta_flat = pm.Dirichlet("theta_flat", a=a_flat, shape=(16, 3))
    obs_flat = pm.Multinomial("obs_flat", n=n_vec, p=theta_flat, observed=counts)
    
    national_bush_flat = pm.Deterministic("national_bush_flat", pm.math.dot(W_norm, theta_flat[:, 0]))
    national_dukakis_flat = pm.Deterministic("national_dukakis_flat", pm.math.dot(W_norm, theta_flat[:, 1]))
    national_lead_flat = pm.Deterministic("national_lead_flat", national_bush_flat - national_dukakis_flat)
    
    idata_flat = pm.sample(draws=3000, tune=1500, chains=4, random_seed=42, return_inferencedata=True)

az.summary(idata_flat, var_names=["national_bush_flat", "national_dukakis_flat", "national_lead_flat"])


### 3. The Opposite Extreme: Complete Pooling
What if we pool all 1,447 respondents into a single nationwide urn, assuming between-stratum variation is zero ($	au = 0$)?
- **Prior:** $	heta \sim 	ext{Dirichlet}(1, 1, 1)$ (3 pseudo-voters nationwide)
- **Likelihood:** $y_{	ext{pooled}} = \sum y_j \sim 	ext{Multinomial}(n=1{,}447, 	heta)$


In [ ]:
y_pooled = counts.sum(axis=0)
n_total = y_pooled.sum()
print(f"Pooled counts (Bush, Dukakis, Other): {y_pooled}, Total n: {n_total}")

with pm.Model() as model_complete_pooling:
    theta_pooled = pm.Dirichlet("theta_pooled", a=np.ones(3))
    obs_pooled = pm.Multinomial("obs_pooled", n=n_total, p=theta_pooled, observed=y_pooled)
    lead_pooled = pm.Deterministic("lead_pooled", theta_pooled[0] - theta_pooled[1])
    
    idata_pooled = pm.sample(draws=3000, tune=1500, chains=4, random_seed=42, return_inferencedata=True)

az.summary(idata_pooled, var_names=["theta_pooled", "lead_pooled"])


### 4. Graphical Models: Plate Notation Comparison
We compare the DAG architectures:
- **Complete Pooling:** $	heta$ is outside the stratum plate ($	au = 0$).
- **No Pooling:** $	heta_j$ is inside the stratum plate ($	au = \infty$).


In [ ]:
# Visualize DAGs using PyMC Graphviz integration
try:
    display(pm.model_to_graphviz(model_complete_pooling))
    display(pm.model_to_graphviz(model_nopooling))
except Exception as e:
    print("Graphviz display skipped (install graphviz system binary for interactive DAGs):", e)


### 5. Visualizing Stratum Uncertainty and the National Margin
We plot the 16 stratum credible intervals side-by-side with the national poststratified margin.


In [ ]:
theta_corr = idata.posterior["theta"].values.reshape(-1, 16, 3)
theta_flat_arr = idata_flat.posterior["theta_flat"].values.reshape(-1, 16, 3)
lead_corr = idata.posterior["national_lead"].values.flatten() * 100
lead_flat_arr = idata_flat.posterior["national_lead_flat"].values.flatten() * 100

fig = plt.figure(figsize=(14, 8), dpi=150)
gs = gridspec.GridSpec(1, 2, width_ratios=[1.35, 1], wspace=0.28)

# Panel A
ax1 = fig.add_subplot(gs[0])
y_pos = np.arange(16)[::-1]

for j in range(16):
    y = y_pos[j]
    b_corr = theta_corr[:, j, 0] * 100
    b_flat_val = theta_flat_arr[:, j, 0] * 100
    ci_low, ci_high = np.percentile(b_corr, [2.5, 97.5])
    
    ax1.plot([ci_low, ci_high], [y, y], color="#0284c7", lw=2.2, alpha=0.85)
    ax1.scatter(np.median(b_corr), y, color="#0369a1", s=40, zorder=4, label="a_k = 1/16" if j == 0 else "")
    ax1.scatter(np.median(b_flat_val), y, color="#f59e0b", s=30, marker="s", zorder=4, label="a_k = 1" if j == 0 else "")
    ax1.scatter(counts[j, 0] / n_vec[j] * 100, y, facecolors="none", edgecolors="#64748b", s=35, zorder=4, label="Raw Sample" if j == 0 else "")
    ax1.text(78, y, f"n={n_vec[j]}", va="center", ha="left", fontsize=8.5, color="#64748b")

ax1.axvline(np.median(idata.posterior["national_bush"].values) * 100, color="#0f172a", linestyle="--", lw=1.2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f"{j+1}. {strata_names[j]}" for j in range(16)], fontsize=9)
ax1.set_xlabel("Bush Preference Share (%)", fontweight="bold")
ax1.set_xlim(12, 85)
ax1.set_title("A. Stratum Posteriors (No Pooling) with 95% Credible Intervals", fontweight="bold")
ax1.legend(loc="lower left", fontsize=8)
ax1.grid(True, linestyle=":", alpha=0.6)

# Panel B
ax2 = fig.add_subplot(gs[1])
x_grid = np.linspace(-2, 22, 500)
kde_c = gaussian_kde(lead_corr)(x_grid)
kde_f = gaussian_kde(lead_flat_arr)(x_grid)

ax2.plot(x_grid, kde_c, color="#0284c7", lw=2.5, label=f"a_k = 1/16 (Median: +{np.median(lead_corr):.1f}%)")
ax2.fill_between(x_grid, kde_c, color="#0284c7", alpha=0.2)
ax2.plot(x_grid, kde_f, color="#f59e0b", lw=2, linestyle="--", label=f"a_k = 1 (Median: +{np.median(lead_flat_arr):.1f}%)")
ax2.axvline(0, color="#e11d48", linestyle=":", lw=1.5, label="Tie (0%)")

ax2.set_xlabel("National Bush Lead Margin (%)", fontweight="bold")
ax2.set_ylabel("Posterior Density", fontweight="bold")
ax2.set_title("B. Poststratified National Lead Posterior", fontweight="bold")
ax2.legend(loc="upper right", fontsize=8.5)
ax2.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


### 4. Transitioning to Option C: Hierarchical Partial Pooling
In the unpooled model, Stratum 1 ($n=47$) shows Dukakis winning $62\%$ to $30\%$, and Stratum 9 ($n=21$) shows Bush winning $67\%$ to $29\%$. Because their sample sizes are small, random noise dominates.

To borrow strength across strata without erasing local geographic culture, we map the 3-category simplex to the unrestricted Cartesian plane using Gelman's coordinate transformation (BDA3 Eqs. 8.8–8.9):
$$\beta_{1j} = \text{logit}\left(\frac{\theta_{1j}}{\theta_{1j}+\theta_{2j}}\right), \qquad \beta_{2j} = \text{logit}(1 - \theta_{3j})$$
And assign the hierarchical bivariate Gaussian prior:
$$\begin{pmatrix} \beta_{1j} \\ \beta_{2j} \end{pmatrix} \sim \mathcal{N}_2\left(\begin{pmatrix} \mu_1 \\ \mu_2 \end{pmatrix}, \; \Sigma\right), \quad j = 1, \dots, 16$$


In [ ]:
# 5. PyMC Hierarchical Partial-Pooling Model (BDA3 Section 8.3)
with pm.Model() as model_hierarchical:
    # Level 1: National Hyperpriors
    mu = pm.Normal("mu", mu=0.0, sigma=2.0, shape=2)
    chol, corr, stds = pm.LKJCholeskyCov(
        "chol_cov", n=2, eta=2.0, sd_dist=pm.HalfNormal.dist(sigma=1.0), compute_corr=True
    )

    # Level 2: Non-centered Stratum Logit Coordinates (j = 1 .. 16)
    z = pm.Normal("z", mu=0.0, sigma=1.0, shape=(16, 2))
    beta = pm.Deterministic("beta", mu + pm.math.dot(z, chol.T))

    # Deterministic Inverse-Logits
    alpha_1 = pm.Deterministic("alpha_1", pm.math.invlogit(beta[:, 0]))  # Bush share among decided
    alpha_2 = pm.Deterministic("alpha_2", pm.math.invlogit(beta[:, 1]))  # Decided voter fraction

    # Simplex Recombination
    theta_1 = pm.Deterministic("theta_1", alpha_1 * alpha_2)             # Bush
    theta_2 = pm.Deterministic("theta_2", (1.0 - alpha_1) * alpha_2)       # Dukakis
    theta_3 = pm.Deterministic("theta_3", 1.0 - alpha_2)               # Undecided
    theta = pm.Deterministic("theta", pm.math.stack([theta_1, theta_2, theta_3], axis=-1))

    # Ballot Likelihood
    obs = pm.Multinomial("obs", n=n_vec, p=theta, observed=counts)

    # Level 3: Finite-Population Poststratification Target
    national_bush = pm.Deterministic("national_bush", pm.math.dot(W_norm, theta_1))
    national_duk = pm.Deterministic("national_duk", pm.math.dot(W_norm, theta_2))
    national_alpha1 = pm.Deterministic("national_alpha1", national_bush / (national_bush + national_duk))
    national_lead = pm.Deterministic("national_lead", 2.0 * national_alpha1 - 1.0)

    print("Sampling Hierarchical Model with NUTS...")
    idata_hier = pm.sample(draws=2500, tune=1500, chains=4, random_seed=42, target_accept=0.95)


In [ ]:
# 6. Decompressing BDA3 Table 8.3: Empirical Shrinkage Table
alpha1_draws = idata_hier.posterior["alpha_1"].values.reshape(-1, 16)
hier_meds = np.median(alpha1_draws, axis=0) * 100
hier_low = np.percentile(alpha1_draws, 2.5, axis=0) * 100
hier_high = np.percentile(alpha1_draws, 97.5, axis=0) * 100
raw_decided = counts[:, 0] / (counts[:, 0] + counts[:, 1]) * 100

table_rows = []
for j in range(16):
    table_rows.append({
        "Stratum": j + 1,
        "Region": strata_names[j],
        "n_j": n_vec[j],
        "Raw_Bush_%": f"{raw_decided[j]:.1f}%",
        "Posterior_Median_%": f"{hier_meds[j]:.1f}%",
        "95%_CI": f"[{hier_low[j]:.1f}%, {hier_high[j]:.1f}%]",
        "Shrinkage_Delta": f"{hier_meds[j] - raw_decided[j]:+.1f}%"
    })

df_table83 = pd.DataFrame(table_rows)
print(df_table83.to_string(index=False))

hier_nat_lead = idata_hier.posterior["national_lead"].values.flatten() * 100
print(f"\nNational Bush Lead (Hierarchical Bayes): +{np.median(hier_nat_lead):.1f}% [{np.percentile(hier_nat_lead, 2.5):.1f}%, {np.percentile(hier_nat_lead, 97.5):.1f}%]")


In [ ]:
# 7. Figure 8.4: Shrinkage Forest Plot & Poststratified Lead Density
fig = plt.figure(figsize=(15, 9), dpi=150)
gs = gridspec.GridSpec(1, 2, width_ratios=[1.4, 1.0], wspace=0.30)

ax1 = fig.add_subplot(gs[0])
y_pos = np.arange(16)[::-1]
nat_mean_val = np.median(idata_hier.posterior["national_alpha1"].values) * 100

for j in range(16):
    y = y_pos[j]
    raw_val = raw_decided[j]
    post_val = hier_meds[j]
    
    ax1.plot([hier_low[j], hier_high[j]], [y, y], color="#0284c7", lw=2.5, alpha=0.9)
    if abs(post_val - raw_val) > 1.2:
        ax1.annotate("", xy=(post_val, y), xytext=(raw_val, y),
                     arrowprops=dict(arrowstyle="->", color="#e11d48" if post_val > raw_val else "#2563eb", lw=1.5))
    ax1.scatter(raw_val, y, facecolors="white", edgecolors="#64748b", s=45, linewidth=1.3, label="Raw Sample" if j == 0 else "")
    ax1.scatter(post_val, y, color="#0284c7", s=55, label="Posterior Median" if j == 0 else "")
    ax1.text(78, y, f"n={n_vec[j]}", va="center", fontsize=8.5, color="#64748b")

ax1.axvline(nat_mean_val, color="#0f172a", linestyle="--", lw=1.2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([f"{j+1}. {strata_names[j]}" for j in range(16)], fontsize=9)
ax1.set_xlabel("Bush Share Among Decided Voters (%)", fontweight="bold")
ax1.set_title("A. Hierarchical Shrinkage Across 16 Strata", fontweight="bold")
ax1.legend(loc="lower left", fontsize=8)
ax1.grid(True, linestyle=":", alpha=0.6)

ax2 = fig.add_subplot(gs[1])
x_grid = np.linspace(-2, 22, 500)
kde_h = gaussian_kde(hier_nat_lead)(x_grid)
b_np = idata.posterior["national_bush"].values.flatten()
d_np = idata.posterior["national_dukakis"].values.flatten()
np_decided_lead = (2.0 * (b_np / (b_np + d_np)) - 1.0) * 100
kde_n = gaussian_kde(np_decided_lead)(x_grid)

ax2.plot(x_grid, kde_n, color="#64748b", lw=2, linestyle="--", label=f"No Pooling (Median: +{np.median(np_decided_lead):.1f}%)")
ax2.plot(x_grid, kde_h, color="#059669", lw=2.5, label=f"Hierarchical Bayes (Median: +{np.median(hier_nat_lead):.1f}%)")
ax2.fill_between(x_grid, kde_h, color="#059669", alpha=0.2)
ax2.axvline(0, color="#e11d48", linestyle=":", lw=1.5, label="Tie Margin (0%)")

ax2.set_xlabel("Bush Lead Margin Among Decided Voters (%)", fontweight="bold")
ax2.set_ylabel("Posterior Density", fontweight="bold")
ax2.set_title("B. Poststratified National Election Margin", fontweight="bold")
ax2.legend(loc="upper right", fontsize=8.5)
ax2.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# 8. Model Checking: Gelman's Posterior Predictive Test (BDA3 p. 209-210)
ppc = pm.sample_posterior_predictive(idata_hier, random_seed=42)
y_rep = ppc.posterior_predictive["obs"].values.reshape(-1, 16, 3)

# Test Statistic: Minimum proportion of Bush among decided across the 16 strata
t_obs = np.min(counts[:, 0] / (counts[:, 0] + counts[:, 1]))
decided_rep = np.maximum(y_rep[:, :, 0] + y_rep[:, :, 1], 1)
t_rep = np.min(y_rep[:, :, 0] / decided_rep, axis=1)

print(f"Observed Test Statistic T(y) = min_j y_{{1j}}/(y_{{1j}}+y_{{2j}}): {t_obs:.3f} (32.6% in Stratum 1)")
print(f"Replicated T(y_rep) Median: {np.median(t_rep):.3f}")
print(f"Replicated T(y_rep) 95% Interval: [{np.percentile(t_rep, 2.5):.3f}, {np.percentile(t_rep, 97.5):.3f}]")
print(f"Posterior Predictive p-value: {np.mean(t_rep >= t_obs):.3f}")


### 9. Final Synthesis
1. **The Fluke Correction:** Strata 1 and 5 were oversampled rural buckets with small sample sizes ($n=47$) showing heavy Dukakis margins. The hierarchical model recognized their small sample sizes could not sustain such extreme deviation, pulling them toward the national average ($51\%$).
2. **National Lead Expansion:** By removing the artificial drag of these two oversampled flukes, the poststratified national Bush lead expands from $+9.8\%$ to **$+11.0\%$**.
3. **No Over-Shrinkage:** Gelman's posterior predictive test proves that observing a minimum vote share as low as $32.6\%$ in 16 small cells is completely expected under binomial sampling noise ($p \approx 0.91$).
